# What each model predicts (NBA)

The served win-prob model is an **ensemble** — a standardizing **logistic** (linear, well-calibrated) + a **gradient-boosted tree** (HistGBT, captures interactions), blended 50/50. This notebook shows, per game, what *each* learner predicts alongside the **market** line and the **actual** outcome — so you can see where the models agree, where they diverge, and who's right when they do.

Mirrors `pipelines/train.py` exactly. Predictions are **out-of-sample** (fit on earlier games, shown on later ones) — no leakage.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from train_eval_duckdb import load_events, build_market_pit, _pipeline, K, HFA, DEFAULT_DB
from sportsball.pipelines._elo import walk_forward
from sportsball.quant import features as feat
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
# Build features (walk_forward emits one row per game, in order -> aligns to source rows for labels)
raw = load_events(str(DEFAULT_DB))
market_pit = build_market_pit(raw)
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_,_hc,_ac) in raw]
frows, _ = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=90,
                        form_window=10, market_pit=market_pit)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
meta = pd.DataFrame({'date': pd.to_datetime([r[0] for r in raw], errors='coerce'),
                     'home': [r[1] for r in raw], 'away': [r[2] for r in raw]})
split = 0.85; cut = int(len(X)*split)
print(f'{len(X)} games | train {cut} | test {len(X)-cut}')

In [ ]:
# Fit the two learners on the chronological train split (production config)
logit = _pipeline().fit(X[:cut], y[:cut])
gbt = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
                                     random_state=0).fit(X[:cut], y[:cut])
p_logit = logit.predict_proba(X[cut:])[:,1]
p_gbt = gbt.predict_proba(X[cut:])[:,1]
p_ens = 0.5*p_logit + 0.5*p_gbt
mkt_col = feat.FEATURE_ORDER.index('market_logit')
mlogit = X[cut:, mkt_col]
p_mkt = np.where(mlogit!=0, 1/(1+np.exp(-mlogit)), np.nan)   # NaN where no line
g = meta.iloc[cut:].copy()
g['P_market']=p_mkt; g['P_logistic']=p_logit; g['P_gbt']=p_gbt; g['P_ensemble']=p_ens
g['actual']=y[cut:]
g['label']=g['away']+' @ '+g['home']+'  '+g['date'].dt.strftime('%Y-%m-%d')
g.tail(8)[['label','P_market','P_logistic','P_gbt','P_ensemble','actual']]

## Per-model accuracy & log-loss (out-of-sample)

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
rows=[]
for name,p in [('market',p_mkt),('logistic',p_logit),('GBT',p_gbt),('ensemble',p_ens)]:
    m = ~np.isnan(p)
    rows.append({'model':name,'n':int(m.sum()),
                 'accuracy':round(accuracy_score(y[cut:][m],(p[m]>=0.5).astype(int)),4),
                 'log_loss':round(log_loss(y[cut:][m],p[m],labels=[0,1]),4)})
pd.DataFrame(rows)

## What each model predicts — most recent 25 games

Each row is a game; dots are the models' P(home win); the ★ is the actual result (0 = away won, 1 = home won). Dashed line = 50%. Spread across the dots = disagreement.

In [ ]:
recent = g.dropna(subset=['date']).sort_values('date').tail(25).reset_index(drop=True)
ys = np.arange(len(recent))
series = [('market','P_market','o','gray'),('logistic','P_logistic','s','steelblue'),
          ('GBT','P_gbt','^','darkorange'),('ensemble','P_ensemble','D','crimson')]
fig, ax = plt.subplots(figsize=(11, 10))
for name,col,mk,color in series:
    ax.scatter(recent[col], ys, marker=mk, color=color, s=45, label=name, zorder=3)
ax.scatter(recent['actual'], ys, marker='*', color='black', s=160, label='actual', zorder=4)
ax.axvline(0.5, color='k', lw=.8, ls='--')
ax.set_yticks(ys); ax.set_yticklabels(recent['label'], fontsize=8)
ax.set_xlabel('P(home win)'); ax.set_xlim(-0.05,1.05); ax.invert_yaxis()
ax.legend(loc='lower center', ncol=5, bbox_to_anchor=(0.5,1.01))
ax.set_title('What each model predicts (last 25 games)', pad=28); plt.tight_layout(); plt.show()

## Where logistic and GBT disagree

Each point a test game; off the diagonal = the two learners diverge. Color = which side each picks (agree vs disagree on the >50% pick).

In [ ]:
agree_pick = (p_logit>=0.5) == (p_gbt>=0.5)
plt.figure(figsize=(6.5,6.5))
plt.scatter(p_logit[agree_pick], p_gbt[agree_pick], s=8, alpha=.25, color='steelblue', label='agree on pick')
plt.scatter(p_logit[~agree_pick], p_gbt[~agree_pick], s=12, alpha=.5, color='crimson', label='disagree on pick')
plt.plot([0,1],[0,1],'k--'); plt.axhline(.5,color='gray',lw=.5); plt.axvline(.5,color='gray',lw=.5)
plt.xlabel('logistic P(home)'); plt.ylabel('GBT P(home)')
plt.title('Logistic vs GBT'); plt.legend(); plt.show()
print(f'disagree on the pick in {(~agree_pick).mean()*100:.1f}% of test games')

## Who wins the disagreements?

When the two learners pick different sides, whose pick is right more often — and does blending them (the ensemble) beat either alone on those games?

In [ ]:
yte = y[cut:]
d = ~agree_pick
def acc(p, mask): return accuracy_score(yte[mask], (p[mask]>=0.5).astype(int))
print(f'games where logistic & GBT disagree: {d.sum()} of {len(yte)}')
if d.sum():
    print(f'  logistic accuracy on those: {acc(p_logit,d):.3f}')
    print(f'  GBT      accuracy on those: {acc(p_gbt,d):.3f}')
    print(f'  ensemble accuracy on those: {acc(p_ens,d):.3f}')
print(f'when they AGREE ({(~d).sum()} games): accuracy {acc(p_logit,~d):.3f}')

## Biggest disagreements (test set)

In [ ]:
g2 = g.copy(); g2['gap']=(g2['P_logistic']-g2['P_gbt']).abs()
g2.sort_values('gap',ascending=False).head(12)[['label','P_market','P_logistic','P_gbt','P_ensemble','actual','gap']]